In [2]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

In [3]:
# Estados computacionais
zero = np.array([1, 0], dtype=complex)
one  = np.array([0, 1], dtype=complex)


def ketbra(psi):
    """
    Constrói o operador |psi><psi|.
    """
    return np.outer(psi, psi.conj())


# Estados de Bell
phi_plus = (np.kron(zero, zero) + np.kron(one, one)) / np.sqrt(2)

rho_phi_plus = ketbra(phi_plus)

rho_phi_plus

array([[0.5+0.j, 0. +0.j, 0. +0.j, 0.5+0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0.5+0.j, 0. +0.j, 0. +0.j, 0.5+0.j]])

In [4]:
def verifica_densidade(rho, tol=1e-10):
    """
    Verifica as condições básicas para uma matriz densidade:
    
    rho = rho^\dagger
    rho >= 0
    Tr(rho) = 1
    """
    
    hermitiana = np.allclose(rho, rho.conj().T, atol=tol)
    autovalores = np.linalg.eigvalsh(rho)
    positiva = np.all(autovalores >= -tol)
    normalizada = np.isclose(np.trace(rho), 1, atol=tol)
    
    return {
        "hermitiana": hermitiana,
        "positiva": positiva,
        "traco_1": normalizada,
        "autovalores": autovalores
    }


verifica_densidade(rho_phi_plus)

<>:5: SyntaxWarning: invalid escape sequence '\d'
<>:5: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_56730/1935040699.py:5: SyntaxWarning: invalid escape sequence '\d'
  rho = rho^\dagger


{'hermitiana': True,
 'positiva': np.True_,
 'traco_1': np.True_,
 'autovalores': array([0., 0., 0., 1.])}

In [5]:
def tensor(*args):
    """
    Produto tensorial de vários operadores/estados.
    """
    resultado = args[0]
    
    for estado in args[1:]:
        resultado = np.kron(resultado, estado)
        
    return resultado

In [6]:
psi_000 = tensor(zero, zero, zero)

psi_000

array([1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j])

In [7]:
def ghz(theta=0):
    """
    Estado GHZ de três qubits:
    
    |GHZ_theta> = (|000> + exp(i theta)|111>) / sqrt(2)
    """
    
    estado = (
        tensor(zero, zero, zero)
        + np.exp(1j * theta) * tensor(one, one, one)
    ) / np.sqrt(2)
    
    return estado


psi_ghz = ghz(theta=0)

rho_ghz = ketbra(psi_ghz)

verifica_densidade(rho_ghz)

{'hermitiana': True,
 'positiva': np.True_,
 'traco_1': np.True_,
 'autovalores': array([0., 0., 0., 0., 0., 0., 0., 1.])}

In [8]:
def partial_trace(rho, trace_out, dims=None):
    """
    Calcula o traço parcial sobre os subsistemas indicados.

    Parâmetros
    ----------
    rho : ndarray
        Matriz densidade do sistema completo.

    trace_out : list
        Índices dos subsistemas sobre os quais realizar
        o traço parcial.

        Exemplo para ABC:
            [2]    -> Tr_C(rho) = rho_AB
            [1]    -> Tr_B(rho) = rho_AC
            [1, 2] -> Tr_BC(rho) = rho_A

    dims : list
        Dimensão de cada subsistema.
        Por padrão, todos são qubits.
    """

    if dims is None:
        n = int(np.log2(rho.shape[0]))
        dims = [2] * n

    n = len(dims)

    trace_out = sorted(trace_out, reverse=True)

    tensor_rho = rho.reshape(dims + dims)

    for i in trace_out:
        tensor_rho = np.trace(
            tensor_rho,
            axis1=i,
            axis2=i + len(dims)
        )

        dims.pop(i)

    dim_final = int(np.prod(dims))

    return tensor_rho.reshape(dim_final, dim_final)

In [38]:
rho_AB = partial_trace(rho_ghz, trace_out=[2])

rho_AB

array([[0.5+0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0.5+0.j]])

In [40]:
rho_AC = partial_trace(rho_ghz, trace_out=[1])

rho_AC

array([[0.5+0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0.5+0.j]])

In [41]:
rho_BC = partial_trace(rho_ghz, trace_out=[0])

rho_BC

array([[0.5+0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0.5+0.j]])

In [42]:
print("AB = AC:", np.allclose(rho_AB, rho_AC))
print("AB = BC:", np.allclose(rho_AB, rho_BC))

AB = AC: True
AB = BC: True


In [44]:
rho_A = partial_trace(rho_ghz, trace_out=[1,2])
rho_B = partial_trace(rho_ghz, trace_out=[0,2])
rho_C = partial_trace(rho_ghz, trace_out=[0,1])

print("rho_A =")
print(rho_A)

print("\nrho_B =")
print(rho_B)

print("\nrho_C =")
print(rho_C)

rho_A =
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]

rho_B =
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]

rho_C =
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]


In [45]:
def espectro(rho):
    """
    Retorna os autovalores de uma matriz Hermitiana
    em ordem crescente.
    """
    return np.linalg.eigvalsh(rho)

In [46]:
print("Espectro de rho_A:")
print(espectro(rho_A))

print("\nEspectro de rho_AB:")
print(espectro(rho_AB))

Espectro de rho_A:
[0.5 0.5]

Espectro de rho_AB:
[0.  0.  0.5 0.5]


In [22]:
def estado_de_X(X):
    rho = X @ X.conj().T
    rho = rho / np.trace(rho)
    
    return rho

In [23]:
def erro(X, rho_AB, rho_AC):

    rho = estado_de_X(X)

    rho_AB_calc = partial_trace(
        rho,
        trace_out=[2]
    )

    rho_AC_calc = partial_trace(
        rho,
        trace_out=[1]
    )

    erro_AB = np.linalg.norm(
        rho_AB_calc - rho_AB,
        'fro'
    )**2

    erro_AC = np.linalg.norm(
        rho_AC_calc - rho_AC,
        'fro'
    )**2

    return erro_AB + erro_AC

In [25]:
def X_para_vetor(X):

    return np.concatenate([
        X.real.ravel(),
        X.imag.ravel()
    ])
def vetor_para_X(x):

    N = 8 * 8

    parte_real = x[:N].reshape(8, 8)
    parte_imag = x[N:].reshape(8, 8)

    return parte_real + 1j * parte_imag

In [26]:
rng = np.random.default_rng(1234)

X_inicial = (
    rng.normal(size=(8, 8))
    + 1j * rng.normal(size=(8, 8))
)

In [27]:
x_inicial = X_para_vetor(X_inicial)

In [28]:
def erro_otimizacao(x, rho_AB, rho_AC):

    X = vetor_para_X(x)

    return erro(X, rho_AB, rho_AC)

In [29]:
from scipy.optimize import minimize

In [30]:
psi_GHZ = (
    tensor(zero, zero, zero)
    + tensor(one, one, one)
) / np.sqrt(2)

rho_GHZ = ketbra(psi_GHZ)

rho_AB = partial_trace(
    rho_GHZ,
    trace_out=[2]
)

rho_AC = partial_trace(
    rho_GHZ,
    trace_out=[1]
)

In [31]:
resultado = minimize(
    erro_otimizacao,
    x_inicial,
    args=(rho_AB, rho_AC),
    method="BFGS"
)

In [32]:
x_final = resultado.x

In [33]:
X_final = vetor_para_X(x_final)

In [34]:
rho_reconstruido = estado_de_X(X_final)

In [35]:
rho_AB_reconstruido = partial_trace(
    rho_reconstruido,
    trace_out=[2]
)

rho_AC_reconstruido = partial_trace(
    rho_reconstruido,
    trace_out=[1]
)

In [36]:
erro_AB = np.linalg.norm(
    rho_AB_reconstruido - rho_AB,
    'fro'
)

erro_AC = np.linalg.norm(
    rho_AC_reconstruido - rho_AC,
    'fro'
)

print("Erro AB =", erro_AB)
print("Erro AC =", erro_AC)

Erro AB = 0.0012179328525994397
Erro AC = 0.0009259243263194007


In [37]:
print("Traço =", np.trace(rho_reconstruido))

print(
    "Autovalores =",
    np.linalg.eigvalsh(rho_reconstruido)
)

Traço = (1-1.925929944387236e-34j)
Autovalores = [0.     0.     0.     0.0002 0.0003 0.0008 0.3533 0.6454]
